# Saved checkpoint semantic diagnostic

Select a CPU runtime. Put the updated repository or the extracted code bundle at `/content/Health-Insurance-SLM-Colab`. The first cell runs the offline preflight against the original Drive artifacts. Review the report before authorizing any paid judgments.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/Health-Insurance-SLM-Colab")
root = Path("/content/drive/MyDrive/HealthInsuranceSLM")
original = root / "colab_l4_balanced_5x200_v3"
saved = root / "colab_l4_balanced_5x200_v3_checkpoint_eval"
output = root / "colab_l4_balanced_5x200_v3_semantic_diagnostic"
os.environ.setdefault("DISTILLATION_LOCAL_LOCK_DIR", "/content/distillation-eval-locks")
command = [sys.executable, "-m", "distillation_loop.semantic_eval",
           "--comparison-dir", str(saved), "--split", str(original / "evaluation_split.json"),
           "--checkpoint-root", str(original), "--historical-run", str(original),
           "--output", str(output)]
cache = original / "benchmark_judging"
if cache.is_dir():
    command += ["--legacy-cache", str(cache)]
subprocess.run(command, cwd=project, check=True)

In [ ]:
# Run only after reviewing cost_preflight.json and approving its cumulative cap.
import json
from getpass import getpass

preflight = json.loads((output / "cost_preflight.json").read_text())
if preflight["unique_missing_judgments"]:
    approval = input("Paste the approved preflight hash (blank cancels): ").strip()
    if approval:
        cap = input("Approved cumulative ceiling in USD: ").strip()
        if approval != preflight["approval_hash"]:
            raise ValueError("Approval does not match the preflight")
        env = dict(os.environ)
        if not env.get("OPENAI_API_KEY"):
            env["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
        subprocess.run(command + ["--approve-preflight", approval, "--max-usd", cap],
                       cwd=project, env=env, check=True)